# 18.2 Data collection and sampling design

Sampling design decides what population your dataset is allowed to represent.

Collection writes the first model assumption into the table. This notebook compares target-mixture sampling, biased sampling, stratified sampling, and weighted evaluation while keeping the classifier fixed.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(18)
random.seed(18)


def dataquality_ladder():
    """D1..D5 over real Breast Cancer with progressively worse data quality."""
    bc = load_breast_cancer()
    X0 = bc.data.astype(float)
    y0 = bc.target
    rng = np.random.default_rng(18)
    rungs = []

    rungs.append(("D1 clean", X0.copy(), y0.copy()))

    y2 = y0.copy()
    flip = rng.random(y2.shape) < 0.15
    y2[flip] = 1 - y2[flip]
    rungs.append(("D2 15% label noise", X0.copy(), y2))

    keep_pos = np.where(y0 == 1)[0]
    keep_neg = np.where(y0 == 0)[0][:30]
    idx = np.concatenate([keep_pos, keep_neg])
    rungs.append(("D3 class imbalance", X0[idx].copy(), y0[idx].copy()))

    X4 = X0 + rng.normal(0.0, X0.std(axis=0) * 0.5, size=X0.shape)
    rungs.append(("D4 heavy feature noise", X4, y0.copy()))

    X5 = X0.copy()
    holes = rng.random(X5.shape) < 0.2
    X5[holes] = np.nan
    col_means = np.nanmean(X5, axis=0)
    X5 = np.where(np.isnan(X5), col_means, X5)
    rungs.append(("D5 20% missing (mean-imputed)", X5, y0.copy()))

    return rungs


def split_scale(X, y, seed=0):
    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    return x_train, x_test, y_train, y_test


def fit_logistic(x_train, y_train, sample_weight=None):
    model = LogisticRegression(max_iter=2000, solver="lbfgs")
    model.fit(x_train, y_train, sample_weight=sample_weight)
    return model


def preview_ladder(rungs):
    for i, (name, X, y) in enumerate(rungs, start=1):
        counts = dict(zip(*np.unique(y, return_counts=True)))
        print(f"{i}. {name}: X={X.shape}, class_counts={counts}")
    print("sample rows", np.round(rungs[0][1][:3, :4], 3).tolist())


def plot_rungs_and_metric(rungs, rows, title, ylabel="accuracy"):
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    metrics = [row["metric"] for row in rows]
    for ax, (name, X, y), metric in zip(axes[0], rungs, metrics):
        ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="coolwarm", alpha=0.65)
        ax.set_title(f"{name}\n{metric:.3f}")
        ax.set_xticks([])
        ax.set_yticks([])
    axes[1][0].plot(range(1, 6), metrics, marker="o")
    axes[1][0].set_title(title)
    axes[1][0].set_xlabel("data-quality rung")
    axes[1][0].set_ylabel(ylabel)
    axes[1][0].set_ylim(0.0, 1.05)
    for ax in axes[1][1:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()



## The concept, built once (D1)

For group sizes $N_g$ and rates $p_g$, the population rate is $p=\sum_g (N_g/N)p_g$. A binomial estimate from $n$ rows has $SE=\sqrt{p(1-p)/n}$.

The key distinction is systematic mixture bias versus random error.

In [ ]:
def design_numbers():
    size_a = 800
    size_b = 200
    rate_a = 0.2
    rate_b = 0.7
    population = (size_a * rate_a + size_b * rate_b) / (size_a + size_b)
    biased = (90 * rate_a + 10 * rate_b) / 100
    stratified = (80 * rate_a + 20 * rate_b) / 100
    se_100 = math.sqrt(population * (1 - population) / 100)
    se_400 = math.sqrt(population * (1 - population) / 400)
    half_width = 1.96 * se_100
    return population, biased, stratified, se_100, se_400, half_width


def sample_design_train(X, y, target_pos_rate=None, sample_pos_rate=None, seed=0):
    rng = np.random.default_rng(seed)
    target_pos_rate = float(y.mean()) if target_pos_rate is None else target_pos_rate
    sample_pos_rate = target_pos_rate if sample_pos_rate is None else sample_pos_rate
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n = min(len(y), 240)
    n_pos = min(len(pos_idx), max(2, int(round(n * sample_pos_rate))))
    n_neg = min(len(neg_idx), max(2, n - n_pos))
    chosen = np.concatenate([
        rng.choice(pos_idx, size=n_pos, replace=False),
        rng.choice(neg_idx, size=n_neg, replace=False),
    ])
    Xs = X[chosen]
    ys = y[chosen]
    weights = np.where(ys == 1, target_pos_rate / max(ys.mean(), 1e-6), (1 - target_pos_rate) / max(1 - ys.mean(), 1e-6))
    x_train, x_test, y_train, y_test = split_scale(Xs, ys, seed=seed)
    w_train, w_test = train_test_split(weights, test_size=0.35, random_state=seed, stratify=ys)
    model = fit_logistic(x_train, y_train, sample_weight=w_train)
    preds = model.predict(x_test)
    return accuracy_score(y_test, preds, sample_weight=w_test)

Assert the lesson's rate, bias, stratification, and uncertainty calculations.

In [ ]:
population, biased, stratified, se_100, se_400, half_width = design_numbers()
print(round(population, 3), round(biased, 3), round(stratified, 3))
print(round(se_100, 4), round(se_400, 4), round(half_width, 4))
assert round(population, 3) == 0.3
assert round(biased, 3) == 0.25
assert round(stratified, 3) == 0.3
assert round(se_100, 4) == 0.0458
assert round(se_400, 4) == 0.0229
assert round(half_width, 4) == 0.0898

## The dataset ladder

Every notebook uses the same real Breast Cancer base table and changes data quality only: clean, label-noisy, imbalanced, feature-noisy, then missing-and-imputed. The model stays fixed so movement in the curve is caused by the data.

In [ ]:
rungs = dataquality_ladder()
preview_ladder(rungs)

In [ ]:
rows = []
for name, X, y in rungs:
    target = float(y.mean())
    metric = sample_design_train(X, y, target_pos_rate=target, sample_pos_rate=target, seed=2)
    rows.append({"name": name, "metric": metric})
    print(f"{name:28s} weighted_accuracy={metric:.3f} target_positive_rate={target:.3f}")

In [ ]:
plot_rungs_and_metric(rungs, rows, "Weighted accuracy vs data quality")

## Pitfall on D5: confusing sample size with representativeness

A large biased sample has small random error but the wrong group mixture. The fix is to evaluate or train with target weights or use a stratified sample matching the intended mixture.

In [ ]:
def unweighted_collection_eval(X, y, sample_pos_rate, seed=4):
    rng = np.random.default_rng(seed)
    x_pool, x_test, y_pool, y_test = train_test_split(X, y, test_size=0.35, random_state=seed, stratify=y)
    pos_idx = np.where(y_pool == 1)[0]
    neg_idx = np.where(y_pool == 0)[0]
    n = min(260, len(y_pool))
    n_pos = min(len(pos_idx), max(2, int(round(n * sample_pos_rate))))
    n_neg = min(len(neg_idx), max(2, n - n_pos))
    chosen = np.concatenate([
        rng.choice(pos_idx, size=n_pos, replace=False),
        rng.choice(neg_idx, size=n_neg, replace=False),
    ])
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_pool[chosen])
    x_eval = scaler.transform(x_test)
    model = fit_logistic(x_train, y_pool[chosen])
    preds = model.predict(x_eval)
    return accuracy_score(y_test, preds)

name, X, y = rungs[-1]
target = float(y.mean())
large_biased = unweighted_collection_eval(X, y, sample_pos_rate=0.98, seed=4)
target_stratified = unweighted_collection_eval(X, y, sample_pos_rate=target, seed=4)
print(f"large biased sample accuracy={large_biased:.3f}")
print(f"target-mixture stratified accuracy={target_stratified:.3f}")
assert target_stratified >= large_biased

## Evaluate it + Practice

- Metric: weighted held-out accuracy under the target mixture; compare with the no-skill majority-class baseline.
- Cheap sanity check: the 80/20 stratified sample should recover the 0.300 population rate.
- Ablation: remove sample weights after deliberate oversampling and compare the metric.
- Failure signals: sample proportions far from the population frame or tiny rare-group test counts.

Practice prompts:
1. Change one corruption level in `dataquality_ladder()` and predict the metric direction.


In [ ]:
# Your turn: change one corruption and rerun the ladder table.


2. Add one slice check that could catch a global-average pitfall.

In [ ]:
# Your turn: define a slice and compare its metric with the global metric.


3. Replace the fixed logistic model with another CPU-safe classifier and explain whether the data-quality ordering changed.

In [ ]:
# Your turn: try a second model without changing the data-cleaning method.
